# Synapsys — Simulators & Solvers Guide

This notebook covers the **physical simulator** layer of synapsys:
nonlinear continuous-time systems, numerical integrators, noise,
linearisation, failure detection, and 2D visualisation.

| Simulator | System | States | Input | Stable? |
|---|---|---|---|---|
| `MassSpringDamperSim` | Linear 1-DOF | `[q, q̇]` | Force F | Always |
| `InvertedPendulumSim` | Nonlinear pivot | `[θ, θ̇]` | Torque τ | No (unstable) |
| `CartPoleSim` | Lagrangian cart-pole | `[p, ṗ, θ, θ̇]` | Force F | No (unstable) |

> **All simulators share the same API:** `reset()`, `step(u, dt)`, `linearize(x0, u0)`, `set_params(**kw)`


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.30,
        "lines.linewidth": 1.8,
        "font.size": 11,
    }
)

---
## 1. Common API — `SimulatorBase`

Every simulator exposes the same interface:

```python
sim = CartPoleSim()           # construct with defaults
y   = sim.reset()             # reset state → returns initial observation
y   = sim.reset(x0=np.zeros(4))  # reset to specific state

y, info = sim.step(u, dt)    # advance one timestep
# info['x']       → full state after step
# info['t_step']  → integration time (= dt)
# info['failed']  → True when system left safe region

ss = sim.linearize(x0, u0)  # numerical Jacobian → StateSpace(A, B, C, D)
sim.set_params(m_c=2.0)     # thread-safe parameter update
```


In [ ]:
from synapsys.simulators import CartPoleSim, InvertedPendulumSim, MassSpringDamperSim

# ── Instantiate all three simulators ─────────────────────────────────────────
msd = MassSpringDamperSim(m=1.0, c=0.5, k=2.0)
pend = InvertedPendulumSim(m=1.0, l=1.0, g=9.81, b=0.1)
cart = CartPoleSim(m_c=1.0, m_p=0.1, l=0.5, g=9.81)

for name, sim in [("MSD", msd), ("Pendulum", pend), ("CartPole", cart)]:
    y0 = sim.reset()
    y1, info = sim.step(np.zeros(sim.input_dim), dt=0.02)
    print(
        f"{name:10s}  state_dim={sim.state_dim}  input_dim={sim.input_dim}"
        f"  output_dim={sim.output_dim}  failed={info['failed']}"
    )

---
## 2. Mass-Spring-Damper

$$m\ddot{q} + c\dot{q} + kq = F$$

Linear, always stable for positive parameters. Useful as a sanity-check
because the analytical solution is known.

The simulator exposes two diagnostic helpers:
- `natural_frequency()` → $\omega_n = \sqrt{k/m}$
- `damping_ratio()` → $\zeta = c / (2\sqrt{km})$


In [ ]:
from scipy.signal import lti
from scipy.signal import step as scipy_step

m, c, k, F = 1.0, 0.5, 2.0, 1.0
dt, T = 0.001, 10.0
steps = int(T / dt)
t = np.arange(steps) * dt

sim = MassSpringDamperSim(m=m, c=c, k=k, integrator="rk4")
print(f"ωₙ = {sim.natural_frequency():.4f} rad/s")
print(
    f"ζ  = {sim.damping_ratio():.4f}  ({'underdamped' if sim.damping_ratio() < 1 else 'overdamped'})"
)
print(f"Steady-state q_ss = F/k = {F / k:.3f} m")

# ── Step response: synapsys vs scipy analytical ───────────────────────────────
sim.reset()
y_sim = [sim.step(np.array([F]), dt)[0][0] for _ in range(steps)]

sys_ct = lti([1 / m], [1, c / m, k / m])
t_ref, y_ref = scipy_step(sys_ct, T=t)
y_ref *= F

# ── Free vibration from x0 = [1, 0] ──────────────────────────────────────────
sim.reset(x0=np.array([1.0, 0.0]))
y_free = [sim.step(np.zeros(1), dt)[0][0] for _ in range(steps)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle(
    f"Mass-Spring-Damper  (m={m}, c={c}, k={k})  ωₙ={sim.natural_frequency():.2f} rad/s  ζ={sim.damping_ratio():.3f}"
)

ax1.plot(t, y_sim, lw=2, label="Synapsys RK4")
ax1.plot(t_ref, y_ref, "--", lw=1.5, label="SciPy (analytical)")
ax1.axhline(F / k, color="gray", ls=":", lw=1, label=f"q_ss = {F / k:.2f} m")
ax1.set_ylabel("Position q (m)")
ax1.set_title("Step Response  (F = 1 N)")
ax1.legend()

ax2.plot(t, y_free, lw=2, color="darkorange", label="Free vibration from x₀=[1, 0]")
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Position q (m)")
ax2.set_title("Free Vibration  (no input)")
ax2.legend()
plt.tight_layout()
plt.show()

rms = np.sqrt(np.mean((np.array(y_sim) - y_ref) ** 2))
print(f"\nRMS error vs scipy: {rms:.2e} m  (should be < 1e-5 for dt=0.001)")

### 2.1 Linearisation — extracting $(A, B, C, D)$

`linearize(x0, u0)` returns a `StateSpace` object via central finite differences.
Since MSD is already linear, the result matches the analytical matrices exactly.


In [ ]:
sim = MassSpringDamperSim(m=m, c=c, k=k)
sim.reset()
ss_msd = sim.linearize(np.zeros(2), np.zeros(1))

print("Numerical Jacobian (linearize):")
print(f"  A =\n{ss_msd.A}")
print(f"  B = {ss_msd.B.flatten()}")
print(f"  C = {ss_msd.C.flatten()}")
print()
print("Analytical (known):")
print(f"  A = [[0, 1], [-k/m, -c/m]] = [[0, 1], [{-k / m:.1f}, {-c / m:.1f}]]")
print(f"  B = [0, 1/m] = [0, {1 / m:.1f}]")
print()
print("Eigenvalues (poles):", np.linalg.eigvals(ss_msd.A).round(4))

---
## 3. Inverted Pendulum

$$I\ddot{\theta} = \frac{g}{l}\sin(\theta) - \frac{b}{I}\dot{\theta} + \frac{\tau}{I}, \qquad I = ml^2$$

Intrinsically unstable: the upright equilibrium ($\theta=0$) has an unstable
eigenvalue $\lambda_+ = +\sqrt{g/l}$. Without control, any small perturbation
causes the pole to fall.


In [ ]:
m, l, g, b = 1.0, 1.0, 9.81, 0.1
dt, T = 0.005, 4.0
steps = int(T / dt)
t = np.arange(steps) * dt

sim = InvertedPendulumSim(m=m, l=l, g=g, b=b)

print(f"Unstable open-loop pole: λ₊ = +{sim.unstable_pole():.4f} rad/s")
print(f"(analytical: √(g/l) = {np.sqrt(g / l):.4f})")

# ── Open-loop: no control, small perturbation → falls ─────────────────────────
sim.reset(x0=np.array([0.05, 0.0]))
theta_ol, fail_t = [], None
for i in range(steps):
    y, info = sim.step(np.zeros(1), dt)
    theta_ol.append(np.degrees(y[0]))
    if info["failed"] and fail_t is None:
        fail_t = t[i]

print(f"\nOpen-loop: failure at t = {fail_t:.2f} s  (|θ| > 90°)")

In [ ]:
from synapsys.algorithms.lqr import lqr

# ── Linearise around upright equilibrium (x0=0, u0=0) ─────────────────────────
sim.reset()
ss_pend = sim.linearize(np.zeros(2), np.zeros(1))

print("Linearised model (around θ=0):")
print(f"  A =\n{ss_pend.A.round(4)}")
print(f"  Open-loop poles: {np.linalg.eigvals(ss_pend.A).round(4)}")

# ── LQR design ────────────────────────────────────────────────────────────────
Q = np.diag([50.0, 1.0])  # penalise angle strongly
R = np.eye(1)
K, P = lqr(ss_pend.A, ss_pend.B, Q, R)

cl_poles = np.linalg.eigvals(ss_pend.A - ss_pend.B @ K)
print(f"  LQR gain K = {K.ravel().round(4)}")
print(f"  Closed-loop poles: {cl_poles.round(4)}")

# ── Closed-loop simulation (larger perturbation) ───────────────────────────────
sim.reset(x0=np.array([0.3, 0.0]))
theta_cl, torque_cl = [], []
for _ in range(steps):
    x = sim.state
    u = np.clip(-K @ x, -20, 20)
    y, _ = sim.step(u, dt)
    theta_cl.append(np.degrees(y[0]))
    torque_cl.append(float(u.flat[0]))

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle("Inverted Pendulum — Open-Loop Fall vs LQR Stabilisation")

ax1.plot(t, theta_ol, color="crimson", lw=1.8, label="Open-loop (θ₀=0.05 rad) — falls")
ax1.plot(t, theta_cl, color="steelblue", lw=1.8, label="LQR (θ₀=0.3 rad) — stabilised")
ax1.axhline(0, color="k", lw=0.8)
ax1.axhline(90, color="gray", ls=":", lw=1, label="Failure threshold ±90°")
ax1.axhline(-90, color="gray", ls=":", lw=1)
ax1.set_ylabel("Angle θ (deg)")
ax1.set_ylim(-110, 110)
ax1.legend()

ax2.plot(t, torque_cl, color="darkorange", lw=1.5, label="LQR torque τ (N·m)")
ax2.axhline(0, color="k", lw=0.8)
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Torque (N·m)")
ax2.legend()
plt.tight_layout()
plt.show()

---
## 4. Cart-Pole

The most complex built-in simulator. Lagrangian dynamics:

$$\Delta = m_c + m_p \sin^2(\theta)$$
$$\ddot{p} = \frac{F + m_p\sin\theta(l\dot\theta^2 - g\cos\theta)}{\Delta}$$
$$\ddot{\theta} = \frac{g\sin\theta - \ddot{p}\cos\theta}{l}$$

**Observation:** `y = [p, θ]` (partial — velocities not observed)  
**Failure:** `|p| > 4.8 m` or `|θ| > π/3 rad (≈60°)`


In [ ]:
m_c, m_p, l, g = 1.0, 0.1, 0.5, 9.81
dt, T = 0.02, 8.0
steps = int(T / dt)
t = np.arange(steps) * dt

sim = CartPoleSim(m_c=m_c, m_p=m_p, l=l, g=g)

# ── Linearise and design LQR ──────────────────────────────────────────────────
sim.reset()
ss_cart = sim.linearize(np.zeros(4), np.zeros(1))

Q = np.diag([1.0, 0.1, 100.0, 10.0])  # strong angle penalty
R = np.eye(1) * 0.01
K, _ = lqr(ss_cart.A, ss_cart.B, Q, R)

print(f"LQR gain K = {K.ravel().round(3)}")
print(f"Closed-loop poles: {np.linalg.eigvals(ss_cart.A - ss_cart.B @ K).round(3)}")

# ── Simulate ──────────────────────────────────────────────────────────────────
x0 = np.array([0.0, 0.0, 0.15, 0.0])  # pole tilted 0.15 rad ≈ 8.6°
sim.reset(x0=x0)

pos, angle, force = [], [], []
for _ in range(steps):
    x = sim.state
    u = np.clip(-K @ x, -50, 50)
    y, info = sim.step(u, dt)
    pos.append(x[0])
    angle.append(np.degrees(x[2]))
    force.append(float(u.flat[0]))
    if info["failed"]:
        print(f"Episode failed at t={_ * dt:.2f} s")
        break

fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
fig.suptitle("Cart-Pole — LQR Stabilisation  (θ₀ = 0.15 rad)")

axes[0].plot(t[: len(pos)], pos, color="steelblue", lw=1.8)
axes[0].axhline(0, color="k", lw=0.8)
axes[0].axhline(4.8, color="gray", ls=":", lw=1, label="±4.8 m limit")
axes[0].axhline(-4.8, color="gray", ls=":", lw=1)
axes[0].set_ylabel("Cart position p (m)")
axes[0].legend()

axes[1].plot(t[: len(angle)], angle, color="darkorange", lw=1.8)
axes[1].axhline(0, color="k", lw=0.8)
axes[1].axhline(60, color="gray", ls=":", lw=1, label="±60° limit")
axes[1].axhline(-60, color="gray", ls=":", lw=1)
axes[1].set_ylabel("Pole angle θ (deg)")
axes[1].legend()

axes[2].plot(t[: len(force)], force, color="seagreen", lw=1.5)
axes[2].axhline(0, color="k", lw=0.8)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Force F (N)")
plt.tight_layout()
plt.show()

### 4.1 Linearised mode vs full nonlinear dynamics

`CartPoleSim(linearised=True)` replaces the Lagrangian ODE with its small-angle
linear approximation. Both should agree near $\theta \approx 0$.


In [ ]:
sim_nl = CartPoleSim(m_c=m_c, m_p=m_p, l=l, g=g, linearised=False)
sim_l = CartPoleSim(m_c=m_c, m_p=m_p, l=l, g=g, linearised=True)

x0_small = np.array([0.0, 0.0, 0.05, 0.0])  # small angle → both agree
x0_large = np.array([0.0, 0.0, 0.35, 0.0])  # large angle → diverge

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, x0, title in zip(
    axes,
    [x0_small, x0_large],
    ["Small angle (θ₀=0.05 rad) — agree", "Large angle (θ₀=0.35 rad) — diverge"],
):
    sim_nl.reset(x0=x0)
    sim_l.reset(x0=x0)
    th_nl, th_l = [], []
    for _ in range(steps):
        u = np.zeros(1)
        y_nl, _ = sim_nl.step(u, dt)
        y_l, _ = sim_l.step(u, dt)
        th_nl.append(np.degrees(y_nl[1]))
        th_l.append(np.degrees(y_l[1]))
        if sim_nl.failed or sim_l.failed:
            break
    n = len(th_nl)
    ax.plot(t[:n], th_nl, lw=2, label="Nonlinear")
    ax.plot(t[:n], th_l, "--", lw=1.5, label="Linearised")
    ax.set_title(title)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Angle θ (deg)")
    ax.legend()

fig.suptitle("Nonlinear vs Linearised CartPoleSim (open-loop, no control)")
plt.tight_layout()
plt.show()

---
## 5. Numerical Integrators (Solvers)

All simulators accept `integrator='euler'|'rk4'|'rk45'` at construction time.

| Method | Order | Cost per step | Use when |
|---|---|---|---|
| `euler` | 1st | 1 function eval | dt ≤ 1 ms, RL inner loops |
| `rk4` | 4th | 4 function evals | **default** — best accuracy/speed tradeoff |
| `rk45` | 4/5th adaptive | 6 function evals | reference / validation |

RK45 is the reference here: we measure how much RK4 and Euler drift from it.


In [ ]:
import time

# ── Setup: LQR on CartPole ────────────────────────────────────────────────────
m_c, m_p, l, g = 1.0, 0.1, 0.5, 9.81
dt_bench = 0.02  # 50 Hz — realistic control rate
T_bench = 5.0
steps_b = int(T_bench / dt_bench)
x0_b = np.array([0.0, 0.0, 0.18, 0.0])

_ref = CartPoleSim(integrator="rk4")
_ref.reset()
ss_b = _ref.linearize(np.zeros(4), np.zeros(1))
K_b, _ = lqr(ss_b.A, ss_b.B, np.diag([1.0, 0.1, 100.0, 10.0]), np.eye(1) * 0.01)


def run_integrator(name):
    sim = CartPoleSim(m_c=m_c, m_p=m_p, l=l, g=g, integrator=name)
    sim.reset(x0=x0_b)
    angles = []
    t0 = time.perf_counter()
    for _ in range(steps_b):
        u = np.clip(-K_b @ sim.state, -50, 50)
        sim.step(u, dt_bench)
        angles.append(np.degrees(sim.state[2]))
    elapsed = time.perf_counter() - t0
    return np.array(angles), elapsed


results = {}
for name in ("rk45", "rk4", "euler"):
    ang, wall = run_integrator(name)
    results[name] = {"angles": ang, "time": wall}

ref = results["rk45"]["angles"]
print(f"{'Integrator':10s}  {'Wall time':>12s}  {'RMS error vs RK45':>20s}")
print("-" * 50)
for name in ("rk45", "rk4", "euler"):
    wall = results[name]["time"] * 1000
    rms = (
        np.sqrt(np.mean((results[name]["angles"] - ref) ** 2)) if name != "rk45" else 0
    )
    print(f"{name:10s}  {wall:10.1f} ms  {rms:20.4f} deg")

In [ ]:
t_b = np.arange(steps_b) * dt_bench
colors = {"rk45": "#ef4444", "rk4": "#3b82f6", "euler": "#f97316"}
labels = {"rk45": "RK45 (reference)", "rk4": "RK4", "euler": "Euler"}

fig, (ax_a, ax_e) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
fig.suptitle(f"Solver Comparison — Cart-Pole LQR  (dt={dt_bench} s, T={T_bench} s)")

for name, c in colors.items():
    ax_a.plot(t_b, results[name]["angles"], color=c, lw=1.5, label=labels[name])
ax_a.axhline(0, color="k", lw=0.8)
ax_a.set_ylabel("Pole angle (deg)")
ax_a.legend()

for name in ("rk4", "euler"):
    err = results[name]["angles"] - ref
    ax_e.plot(t_b, err, color=colors[name], lw=1.2, label=f"{name} error")
ax_e.axhline(0, color="k", lw=0.8)
ax_e.set_xlabel("Time (s)")
ax_e.set_ylabel("Angle error vs RK45 (deg)")
ax_e.legend()
plt.tight_layout()
plt.show()

---
## 6. Noise and Disturbances

Every simulator accepts two stochastic parameters at construction:

| Parameter | Model | Effect |
|---|---|---|
| `noise_std=σ` | Gaussian additive noise on **output** | Sensor uncertainty |
| `disturbance_std=σ` | Gaussian additive noise on **input** | Process / actuator noise |

This section shows how each degrades LQR performance on the inverted pendulum.


In [ ]:
from synapsys.algorithms.lqr import lqr

m, l, g, b = 1.0, 1.0, 9.81, 0.05
dt_n, T_n = 0.005, 6.0
steps_n = int(T_n / dt_n)
t_n = np.arange(steps_n) * dt_n
x0_n = np.array([0.2, 0.0])

# ── Design LQR once on clean model ───────────────────────────────────────────
ref_sim = InvertedPendulumSim(m=m, l=l, g=g, b=b)
ref_sim.reset()
ss_n = ref_sim.linearize(np.zeros(2), np.zeros(1))
K_n, _ = lqr(ss_n.A, ss_n.B, np.diag([80.0, 1.0]), np.eye(1))


def run_noisy(noise_std=0.0, disturbance_std=0.0, seed=42):
    np.random.seed(seed)
    sim = InvertedPendulumSim(
        m=m, l=l, g=g, b=b, noise_std=noise_std, disturbance_std=disturbance_std
    )
    sim.reset(x0=x0_n)
    thetas, failed_at = [], None
    for i in range(steps_n):
        x = sim.state
        u = np.clip(-K_n @ x, -30, 30)
        y, info = sim.step(u, dt_n)
        thetas.append(np.degrees(y[0]))
        if info["failed"] and failed_at is None:
            failed_at = t_n[i]
    return np.array(thetas), failed_at


configs = [
    ("Clean", 0.0, 0.0),
    ("Sensor noise σ=0.02", 0.02, 0.0),
    ("Disturbance σ=0.5", 0.0, 0.5),
    ("Both σ=0.02 / σ=0.3", 0.02, 0.3),
]

fig, ax = plt.subplots(figsize=(12, 5))
colors_n = ["steelblue", "darkorange", "seagreen", "crimson"]

for (label, ns, ds), c in zip(configs, colors_n):
    angles, fail_t = run_noisy(ns, ds)
    n = len(angles)
    suffix = f" [fails @ {fail_t:.2f}s]" if fail_t else ""
    ax.plot(t_n[:n], angles, lw=1.5, color=c, label=label + suffix, alpha=0.85)

ax.axhline(0, color="k", lw=0.8)
ax.axhline(90, color="gray", ls=":", lw=1)
ax.axhline(-90, color="gray", ls=":", lw=1)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Angle θ (deg)")
ax.set_title("LQR Robustness — Effect of Sensor Noise and Process Disturbances")
ax.set_ylim(-120, 120)
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Thread-Safe Parameter Updates — `set_params()`

`set_params(**kw)` is protected by a lock — safe to call from any thread
while the control loop runs in another. This lets you change physical
parameters at runtime (mass, length, gravity) to simulate wear or
model uncertainty.


In [ ]:
sim = InvertedPendulumSim(m=1.0, l=1.0, g=9.81, b=0.05)
sim.reset(x0=np.array([0.15, 0.0]))

ss_t = sim.linearize(np.zeros(2), np.zeros(1))
K_t, _ = lqr(ss_t.A, ss_t.B, np.diag([80.0, 1.0]), np.eye(1))

dt_t, T_t = 0.005, 8.0
steps_t = int(T_t / dt_t)
t_t = np.arange(steps_t) * dt_t

thetas_t, masses = [], []
change_log = []  # (time, old_m, new_m)

# Schedule mass changes at specific simulation steps (no real-time sleep needed)
mass_schedule = {
    int(2.0 / dt_t): 2.0,
    int(4.0 / dt_t): 0.5,
    int(6.0 / dt_t): 1.5,
}

for i in range(steps_t):
    if i in mass_schedule:
        old_m = sim.params.get("m", 1.0)
        new_m = mass_schedule[i]
        sim.set_params(m=new_m)
        change_log.append((t_t[i], old_m, new_m))

    x = sim.state
    u = np.clip(-K_t @ x, -30, 30)
    y, info = sim.step(u, dt_t)
    thetas_t.append(np.degrees(y[0]))
    masses.append(sim.params.get("m", 1.0))
    if info["failed"]:
        print(f"Failed at t={i * dt_t:.2f} s")
        break

print("Parameter changes:")
for t_ev, m_old, m_new in change_log:
    print(f"  t={t_ev:.1f}s  m: {m_old} → {m_new} kg")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
fig.suptitle("Thread-Safe Parameter Updates — LQR under Live Mass Changes")

n_t = len(thetas_t)
ax1.plot(t_t[:n_t], thetas_t, color="steelblue", lw=1.5, label="θ (deg)")
ax1.axhline(0, color="k", lw=0.8)
for t_ev, m_old, m_new in change_log:
    ax1.axvline(t_ev, color="crimson", ls="--", lw=1, alpha=0.7)
    ax1.text(t_ev + 0.05, 15, f"m:{m_old}→{m_new}", color="crimson", fontsize=9)
ax1.set_ylabel("Angle θ (deg)")
ax1.legend()

ax2.plot(t_t[:n_t], masses[:n_t], color="darkorange", lw=1.5, label="mass m (kg)")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("m (kg)")
ax2.legend()
plt.tight_layout()
plt.show()

---
## 8. `CartPole2DView` — 2D Visualisation and Headless Simulation

`CartPole2DView` is a matplotlib-based wrapper that:
- Renders the cart-pole in 2D (works in notebooks, no Qt required)
- Provides built-in LQR controller or accepts a custom one
- Runs headlessly via `simulate()` — returns full telemetry history
- Exports GIF/MP4 via `animate(save=...)`

```python
# Three modes of use:
CartPole2DView().run()                          # interactive window
hist = CartPole2DView(duration=5.0).simulate()  # headless — returns dict
anim = CartPole2DView().animate(save='out.gif') # save animation
```


In [ ]:
import matplotlib

matplotlib.use("Agg")  # headless — no display required

from synapsys.viz import CartPole2DView

# ── Headless simulate() ───────────────────────────────────────────────────────
view = CartPole2DView(dt=0.02, duration=6.0)
hist = view.simulate()

print("History keys:", list(hist.keys()))
print(f"Steps simulated: {len(hist['t'])}")
print(f"Final angle:     {np.degrees(hist['angle'][-1]):.3f} deg")
print(f"Final position:  {hist['pos'][-1]:.4f} m")

t_2d = np.array(hist["t"])
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
fig.suptitle("CartPole2DView — Headless simulate()  (built-in LQR)")

axes[0].plot(t_2d, hist["pos"], color="steelblue", lw=1.8)
axes[0].set_ylabel("Cart pos p (m)")
axes[0].axhline(0, color="k", lw=0.8)

axes[1].plot(t_2d, np.degrees(hist["angle"]), color="darkorange", lw=1.8)
axes[1].set_ylabel("Pole angle θ (deg)")
axes[1].axhline(0, color="k", lw=0.8)

axes[2].plot(t_2d, hist["force"], color="seagreen", lw=1.5)
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Force F (N)")
axes[2].axhline(0, color="k", lw=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Custom controller passed to CartPole2DView ────────────────────────────────
# Aggressive LQR — penalises angle much more strongly
sim_c = CartPoleSim()
sim_c.reset()
ss_c = sim_c.linearize(np.zeros(4), np.zeros(1))
K_aggressive, _ = lqr(
    ss_c.A, ss_c.B, np.diag([0.1, 0.01, 500.0, 50.0]), np.eye(1) * 0.001
)


def my_controller(x: np.ndarray) -> np.ndarray:
    """Aggressive LQR — returns force array [F]."""
    return np.clip(-K_aggressive @ x, -50, 50)


hist_custom = CartPole2DView(controller=my_controller, dt=0.02, duration=6.0).simulate()

# Compare default vs aggressive LQR
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(
    hist["t"], np.degrees(hist["angle"]), lw=1.8, color="steelblue", label="Default LQR"
)
ax.plot(
    hist_custom["t"],
    np.degrees(hist_custom["angle"]),
    lw=1.8,
    color="crimson",
    label="Aggressive LQR",
)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Pole angle θ (deg)")
ax.set_title("Default vs Aggressive LQR — CartPole2DView")
ax.legend()
plt.tight_layout()
plt.show()

# ── Save GIF ──────────────────────────────────────────────────────────────────
# Uncomment to export (requires pillow or ffmpeg):
# anim = CartPole2DView(controller=my_controller).animate(save='/tmp/cartpole2d.gif')
print("Custom controller — final angle:", np.degrees(hist_custom["angle"][-1]), "deg")

---
## 9. Summary

| Feature | API | Notes |
|---|---|---|
| Create simulator | `CartPoleSim(m_c=1.0, ...)` | Defaults are physically reasonable |
| Reset state | `sim.reset()` / `sim.reset(x0=...)` | Always reset before each episode |
| Step | `y, info = sim.step(u, dt)` | `info['failed']` signals episode end |
| Linearise | `ss = sim.linearize(x0, u0)` | Returns `StateSpace(A,B,C,D)` |
| Choose solver | `CartPoleSim(integrator='rk4')` | `euler`/`rk4`/`rk45` |
| Noise | `MassSpringDamperSim(noise_std=0.01)` | Additive Gaussian on output |
| Disturbance | `...disturbance_std=0.05` | Additive Gaussian on input |
| Thread-safe update | `sim.set_params(m=2.0)` | Safe from any thread |
| 2D headless | `CartPole2DView(duration=5).simulate()` | Returns `{t, pos, angle, force, states}` |
| Save animation | `.animate(save='out.gif')` | Requires `pillow` or `ffmpeg` |
